In [1]:
!pip install -q google-genai pydantic pandas

In [5]:
from google.colab import userdata
from google import genai

# Read API key securely from Colab Secrets
API_KEY = userdata.get("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("GEMINI_API_KEY not found. Check Colab Secrets.")

client = genai.Client(api_key=API_KEY)

# Test Gemini Flash
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Reply with exactly: Gemini API connected successfully."
)

print(response.text)

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

In [7]:
import os

required_files = [
    "appointments_clean.csv",
    "followup_notes.csv",
    "clinics.csv"
]

for file in required_files:
    if os.path.exists(file):
        print(f"FOUND: {file}")
    else:
        print(f"MISSING: {file}")

FOUND: appointments_clean.csv
FOUND: followup_notes.csv
FOUND: clinics.csv


In [8]:
import pandas as pd

appointments = pd.read_csv("appointments_clean.csv")
notes = pd.read_csv("followup_notes.csv")
clinics = pd.read_csv("clinics.csv")

print("Appointments:", appointments.shape)
print("Follow-up notes:", notes.shape)
print("Clinics:", clinics.shape)

print("\nFollow-up notes columns:")
print(notes.columns.tolist())

print("\nAppointments columns:")
print(appointments.columns.tolist())

print("\nClinics columns:")
print(clinics.columns.tolist())

Appointments: (990, 16)
Follow-up notes: (30, 3)
Clinics: (8, 6)

Follow-up notes columns:
['note_id', 'appointment_id', 'note_text']

Appointments columns:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']

Clinics columns:
['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']


In [9]:
# Display the first 10 follow-up notes
first_10_notes = notes.head(10).copy()

pd.set_option("display.max_colwidth", None)

display(first_10_notes)

# Check missing or empty note text
print("\nEmpty note count:",
      first_10_notes["note_text"].fillna("").str.strip().eq("").sum())

# Check appointment ID matching
matched = first_10_notes["appointment_id"].isin(
    appointments["appointment_id"]
).sum()

print("Notes matched with appointments:", matched, "/ 10")

,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
1,N002,A0897,Lives far from the clinic and could not find any transport that morning. Says the clinic is too far for a routine check-up.
2,N003,A0526,Patient felt the consultation fee was too high this month and decided to skip. Asked if there is any discount for follow-ups.
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, stayed in bed. Will visit once recovered."
5,N006,A0507,"Manager called an urgent meeting at the office, could not leave work. Asked for an evening slot next time."
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking karni hai."
7,N008,A0222,"City bus strike that day, no auto available near the colony. Distance is a big problem for him."
8,N009,A0977,"Phone was switched off, could not reach the patient after three attempts."
9,N010,A0596,Had a shift change at the factory and could not take leave. Prefers Saturday appointments.



Empty note count: 1
Notes matched with appointments: 10 / 10


In [10]:
from pydantic import BaseModel, Field
from typing import Literal

ReasonCategory = Literal[
    "No reminder",
    "Transport or distance",
    "Financial",
    "Illness",
    "Work or scheduling",
    "Contact issue",
    "Other",
    "Unknown"
]

class AnalysedNote(BaseModel):
    note_id: str
    appointment_id: str

    reason_category: ReasonCategory

    summary: str = Field(
        description="Short summary of the patient's explanation"
    )

    recommended_action: str = Field(
        description="Practical action the clinic should take"
    )

    confidence: Literal["High", "Medium", "Low"]

    is_empty: bool

class NotesAnalysis(BaseModel):
    analyses: list[AnalysedNote]

print("Pydantic schema created successfully!")
print(NotesAnalysis.model_json_schema())

Pydantic schema created successfully!
{'$defs': {'AnalysedNote': {'properties': {'note_id': {'title': 'Note Id', 'type': 'string'}, 'appointment_id': {'title': 'Appointment Id', 'type': 'string'}, 'reason_category': {'enum': ['No reminder', 'Transport or distance', 'Financial', 'Illness', 'Work or scheduling', 'Contact issue', 'Other', 'Unknown'], 'title': 'Reason Category', 'type': 'string'}, 'summary': {'description': "Short summary of the patient's explanation", 'title': 'Summary', 'type': 'string'}, 'recommended_action': {'description': 'Practical action the clinic should take', 'title': 'Recommended Action', 'type': 'string'}, 'confidence': {'enum': ['High', 'Medium', 'Low'], 'title': 'Confidence', 'type': 'string'}, 'is_empty': {'title': 'Is Empty', 'type': 'boolean'}}, 'required': ['note_id', 'appointment_id', 'reason_category', 'summary', 'recommended_action', 'confidence', 'is_empty'], 'title': 'AnalysedNote', 'type': 'object'}}, 'properties': {'analyses': {'items': {'$ref': '

In [11]:
from pydantic import ValidationError

# Test 1: Missing required fields
invalid_response_1 = {
    "note_id": "N001",
    "appointment_id": "A0872",
    "reason_category": "No reminder"
}

# Test 2: Invalid reason category
invalid_response_2 = {
    "note_id": "N002",
    "appointment_id": "A0897",
    "reason_category": "Transportation problem",
    "summary": "Patient could not find transport.",
    "recommended_action": "Offer transport assistance.",
    "confidence": "High",
    "is_empty": False
}

# Validate both responses
for number, response in enumerate(
    [invalid_response_1, invalid_response_2], start=1
):
    try:
        AnalysedNote.model_validate(response)
        print(f"Test {number}: Unexpectedly passed")
    except ValidationError as error:
        print(f"Test {number}: INVALID RESPONSE CAUGHT")
        print(error.errors()[0]["msg"])

print("\nInvalid response testing completed.")

Test 1: INVALID RESPONSE CAUGHT
Field required
Test 2: INVALID RESPONSE CAUGHT
Input should be 'No reminder', 'Transport or distance', 'Financial', 'Illness', 'Work or scheduling', 'Contact issue', 'Other' or 'Unknown'

Invalid response testing completed.


In [12]:
import json

# Prepare the first 10 notes
notes_for_gemini = first_10_notes.copy()

notes_for_gemini["note_text"] = (
    notes_for_gemini["note_text"].fillna("")
)

notes_json = notes_for_gemini.to_dict(orient="records")

# CRAFT prompt
craft_prompt = f"""
CONTEXT:
CareLoop Clinics wants to understand why patients miss appointments.
Analyse the first 10 patient follow-up notes.

ROLE:
You are a healthcare operations data analyst.

ACTION:
For each note:
1. Identify the main reason for the missed appointment.
2. Write a short, factual summary.
3. Recommend a practical clinic action.
4. Assign confidence as High, Medium, or Low.
5. Set is_empty to true if the note contains no text.
6. For empty notes, use reason_category "Unknown".
7. Understand Hindi text as well as English.
8. Do not invent information.

FORMAT:
Return valid JSON only, following this structure:
{json.dumps(NotesAnalysis.model_json_schema(), indent=2)}

TONE:
Professional, neutral, concise and evidence-based.

PATIENT NOTES:
{json.dumps(notes_json, ensure_ascii=False, indent=2)}
"""

print("CRAFT prompt prepared successfully.")
print("Notes included:", len(notes_json))
print("Prompt characters:", len(craft_prompt))

CRAFT prompt prepared successfully.
Notes included: 10
Prompt characters: 4128


In [13]:
import time
import json
from google.genai import types

gemini_success = False

for attempt in range(1, 6):
    try:
        print(f"Gemini attempt {attempt}/5...")

        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents=craft_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            )
        )

        # Validate Gemini's JSON response
        validated_result = NotesAnalysis.model_validate_json(
            response.text
        )

        # Save raw response
        with open("gemini_raw_response.json", "w") as f:
            f.write(response.text)

        # Convert validated results into DataFrame
        analysed_data = [
            item.model_dump()
            for item in validated_result.analyses
        ]

        notes_analysed = pd.DataFrame(analysed_data)
        notes_analysed.to_csv(
            "notes_analysed.csv", index=False
        )

        print("Gemini analysis successful!")
        print("Notes analysed:", len(notes_analysed))
        print("Saved: notes_analysed.csv")
        gemini_success = True
        break

    except Exception as e:
        print("Attempt failed:", str(e)[:250])

        if attempt < 5:
            wait = 2 ** attempt
            print(f"Retrying in {wait} seconds...")
            time.sleep(wait)

if not gemini_success:
    print("\nGemini is still unavailable.")
    print("We can continue with the remaining offline tasks.")


Gemini attempt 1/5...
Gemini analysis successful!
Notes analysed: 10
Saved: notes_analysed.csv


In [14]:
print("Analysed notes:")
display(notes_analysed)

print("\nReason category counts:")
print(notes_analysed["reason_category"].value_counts())

print("\nEmpty notes handled:")
print(notes_analysed[["note_id", "reason_category", "is_empty"]])

print("\nTotal records:", len(notes_analysed))
print("Unique note IDs:", notes_analysed["note_id"].nunique())
print("Missing summaries:", notes_analysed["summary"].isna().sum())

Analysed notes:


,note_id,appointment_id,reason_category,summary,recommended_action,confidence,is_empty
0,N001,A0872,No reminder,Patient forgot the appointment because no reminder message was received; requested rebooking for next week.,"Verify patient notification preferences and ensure automated SMS/call reminders are enabled, then reschedule the visit.",High,False
1,N002,A0897,Transport or distance,"Patient lives far away, lacked morning transport, and considers the clinic too distant for routine check-ups.",Offer telehealth/virtual consultation options for routine visits or assist with community transit resources.,High,False
2,N003,A0526,Financial,Patient skipped due to consultation cost concerns and inquired about follow-up discounts.,"Provide information regarding sliding-scale fees, discounted follow-up packages, or payment plans.",High,False
3,N004,A0926,Unknown,No note details provided.,Initiate follow-up contact with the patient to ascertain the reason for absence and document feedback.,High,True
4,N005,A0534,Illness,"Patient was bedridden with high fever and body ache, intending to visit after recovery.",Check patient recovery status via follow-up call and reschedule once symptoms subside.,High,False
5,N006,A0507,Work or scheduling,Patient had an urgent workplace meeting and requested evening appointment availability.,Reschedule the patient to an evening or extended-hours clinic slot as requested.,High,False
6,N007,A0333,No reminder,Patient forgot the visit due to not receiving an SMS reminder and requested rebooking.,"Confirm mobile contact details, verify SMS delivery configuration, and rebook the appointment.",High,False
7,N008,A0222,Transport or distance,"Patient could not travel due to a city bus strike, lack of auto-rickshaws, and general distance from the clinic.",Explore telemedicine consultation or schedule future visits during non-disrupted transit periods.,High,False
8,N009,A0977,Contact issue,Clinic staff could not reach the patient across three attempts because the phone was switched off.,Send an alternative outreach message (email or postal communication) and request an alternate contact number upon next touchpoint.,High,False
9,N010,A0596,Work or scheduling,"Patient experienced a factory shift change preventing leave, and expressed preference for Saturday slots.",Reschedule the patient for a Saturday clinic session to accommodate their work schedule.,High,False



Reason category counts:
reason_category
No reminder              2
Transport or distance    2
Work or scheduling       2
Financial                1
Unknown                  1
Illness                  1
Contact issue            1
Name: count, dtype: int64

Empty notes handled:
  note_id        reason_category  is_empty
0    N001            No reminder     False
1    N002  Transport or distance     False
2    N003              Financial     False
3    N004                Unknown      True
4    N005                Illness     False
5    N006     Work or scheduling     False
6    N007            No reminder     False
7    N008  Transport or distance     False
8    N009          Contact issue     False
9    N010     Work or scheduling     False

Total records: 10
Unique note IDs: 10
Missing summaries: 0


In [15]:
# Merge analysed notes with appointments
merged = notes_analysed.merge(
    appointments,
    on="appointment_id",
    how="left",
    validate="one_to_one"
)

# Add clinic information
merged = merged.merge(
    clinics,
    on="clinic_id",
    how="left",
    validate="many_to_one"
)

# Check merge results
print("Merged dataset shape:", merged.shape)
print("Matched appointments:", merged["clinic_id"].notna().sum())
print("Matched clinics:", merged["clinic_name"].notna().sum())

# Save merged dataset
merged.to_csv("merged_notes_appointments_clinics.csv", index=False)

# Display important columns
display(
    merged[
        ["note_id", "appointment_id", "clinic_name",
         "city", "reason_category", "no_show"]
    ]
)

Merged dataset shape: (10, 27)
Matched appointments: 10
Matched clinics: 10


,note_id,appointment_id,clinic_name,city,reason_category,no_show
0,N001,A0872,Vaishali Nagar Clinic,Jaipur,No reminder,1
1,N002,A0897,Vijay Nagar Clinic,Indore,Transport or distance,1
2,N003,A0526,Malviya Nagar Clinic,Jaipur,Financial,1
3,N004,A0926,Malviya Nagar Clinic,Jaipur,Unknown,1
4,N005,A0534,Malviya Nagar Clinic,Jaipur,Illness,1
5,N006,A0507,Malviya Nagar Clinic,Jaipur,Work or scheduling,1
6,N007,A0333,Gomti Nagar Care Point,Lucknow,No reminder,1
7,N008,A0222,Hadapsar Care Centre,Pune,Transport or distance,1
8,N009,A0977,Kothrud Family Clinic,Pune,Contact issue,1
9,N010,A0596,Malviya Nagar Clinic,Jaipur,Work or scheduling,1


In [16]:
# Cross-tabulation: reason by clinic
clinic_crosstab = pd.crosstab(
    merged["clinic_name"],
    merged["reason_category"]
)

# Cross-tabulation: reason by city
city_crosstab = pd.crosstab(
    merged["city"],
    merged["reason_category"]
)

print("REASONS BY CLINIC")
display(clinic_crosstab)

print("\nREASONS BY CITY")
display(city_crosstab)

# Save both tables
clinic_crosstab.to_csv("reason_by_clinic.csv")
city_crosstab.to_csv("reason_by_city.csv")

print("\nBoth cross-tabulation files saved successfully.")

REASONS BY CLINIC


reason_category,Contact issue,Financial,Illness,No reminder,Transport or distance,Unknown,Work or scheduling
clinic_name,,,,,,,
Gomti Nagar Care Point,0,0,0,1,0,0,0
Hadapsar Care Centre,0,0,0,0,1,0,0
Kothrud Family Clinic,1,0,0,0,0,0,0
Malviya Nagar Clinic,0,1,1,0,0,1,2
Vaishali Nagar Clinic,0,0,0,1,0,0,0
Vijay Nagar Clinic,0,0,0,0,1,0,0



REASONS BY CITY


reason_category,Contact issue,Financial,Illness,No reminder,Transport or distance,Unknown,Work or scheduling
city,,,,,,,
Indore,0,0,0,0,1,0,0
Jaipur,0,1,1,1,0,1,2
Lucknow,0,0,0,1,0,0,0
Pune,1,0,0,0,1,0,0



Both cross-tabulation files saved successfully.


In [17]:
recommendations = {
    "Vaishali Nagar Clinic": (
        "Verify SMS reminder delivery and confirm patient contact preferences."
    ),
    "Vijay Nagar Clinic": (
        "Explore transport assistance and telehealth for routine visits."
    ),
    "Malviya Nagar Clinic": (
        "Offer evening and weekend appointments, review follow-up pricing, "
        "and contact patients whose reasons are unknown."
    ),
    "Gomti Nagar Care Point": (
        "Improve automated reminders and confirm successful SMS delivery."
    ),
    "Hadapsar Care Centre": (
        "Explore alternative transport options and telehealth appointments."
    ),
    "Kothrud Family Clinic": (
        "Collect alternate contact details and provide additional outreach channels."
    )
}

recommendation_df = pd.DataFrame(
    list(recommendations.items()),
    columns=["clinic_name", "recommendation"]
)

display(recommendation_df)

recommendation_df.to_csv(
    "clinic_specific_recommendations.csv",
    index=False
)

print("Clinic recommendations saved successfully.")

,clinic_name,recommendation
0,Vaishali Nagar Clinic,Verify SMS reminder delivery and confirm patient contact preferences.
1,Vijay Nagar Clinic,Explore transport assistance and telehealth for routine visits.
2,Malviya Nagar Clinic,"Offer evening and weekend appointments, review follow-up pricing, and contact patients whose reasons are unknown."
3,Gomti Nagar Care Point,Improve automated reminders and confirm successful SMS delivery.
4,Hadapsar Care Centre,Explore alternative transport options and telehealth appointments.
5,Kothrud Family Clinic,Collect alternate contact details and provide additional outreach channels.


Clinic recommendations saved successfully.


In [18]:
basic_prompt = f"""
Analyse these 10 patient follow-up notes.
For each note, identify the reason for the missed appointment,
summarise it, and suggest an action for the clinic.

Return JSON containing an analyses list.
Each item should contain:
note_id, appointment_id, reason_category, summary,
recommended_action, confidence, is_empty.

Use the same reason categories as the Pydantic schema.
Do not invent information.

Notes:
{json.dumps(notes_json, ensure_ascii=False)}
"""

try:
    basic_response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=basic_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        )
    )

    basic_result = NotesAnalysis.model_validate_json(
        basic_response.text
    )

    print("Basic prompt analysis successful!")
    print("Basic prompt records:", len(basic_result.analyses))
    print("CRAFT prompt records:", len(validated_result.analyses))

    print("\nPROMPT COMPARISON")
    print("CRAFT: Explicit context, role, action, format and tone.")
    print("Basic: Direct instruction with a short output specification.")
    print("Both outputs passed Pydantic validation.")

    with open("basic_prompt_response.json", "w") as f:
        f.write(basic_response.text)

except Exception as e:
    print("Basic prompt comparison error:", str(e)[:500])

Basic prompt comparison error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


In [19]:
import time

basic_success = False

for attempt in range(1, 4):
    try:
        print(f"Basic prompt attempt {attempt}/3...")

        basic_response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents=basic_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            )
        )

        basic_result = NotesAnalysis.model_validate_json(
            basic_response.text
        )

        with open("basic_prompt_response.json", "w") as f:
            f.write(basic_response.text)

        print("\nBasic prompt analysis successful!")
        print("Records analysed:", len(basic_result.analyses))
        print("CRAFT records:", len(validated_result.analyses))
        print("Basic records:", len(basic_result.analyses))

        basic_success = True
        break

    except Exception as e:
        print("Error:", str(e)[:200])

        if attempt < 3:
            wait = attempt * 15
            print(f"Waiting {wait} seconds before retry...")
            time.sleep(wait)

if not basic_success:
    print("\nBasic prompt comparison still pending.")

Basic prompt attempt 1/3...
Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Waiting 15 seconds before retry...
Basic prompt attempt 2/3...
Error: 18 validation errors for NotesAnalysis
analyses.0.reason_category
  Input should be 'No reminder', 'Transport or distance', 'Financial', 'Illness', 'Work or scheduling', 'Contact issue', 'Other' or 'U
Waiting 30 seconds before retry...
Basic prompt attempt 3/3...
Error: 21 validation errors for NotesAnalysis
analyses.0.reason_category
  Input should be 'No reminder', 'Transport or distance', 'Financial', 'Illness', 'Work or scheduling', 'Contact issue', 'Other' or 'U

Basic prompt comparison still pending.


In [20]:
import json

try:
    raw_basic = json.loads(basic_response.text)

    print("Raw basic response recovered!")

    for item in raw_basic.get("analyses", []):
        print(
            item.get("note_id"),
            "=>",
            item.get("reason_category")
        )

except Exception as e:
    print("Could not recover response:", str(e))

Raw basic response recovered!
N001 => FORGOT_NO_REMINDER
N002 => TRANSPORTATION
N003 => FINANCIAL
N004 => None
N005 => ILLNESS
N006 => WORK_COMMITMENT
N007 => FORGOT_NO_REMINDER
N008 => TRANSPORTATION
N009 => UNREACHABLE
N010 => WORK_COMMITMENT


In [21]:
comparison = {
    "comparison": "CRAFT vs Basic Prompt",
    "notes_analysed": 10,
    "craft_prompt": {
        "valid_records": 10,
        "schema_validation": "Passed",
        "consistent_categories": True
    },
    "basic_prompt": {
        "records_returned": 10,
        "schema_validation": "Failed",
        "category_consistency": "Inconsistent",
        "issues": [
            "Used different category labels",
            "N004 had no reason category",
            "Pydantic rejected the response"
        ]
    },
    "conclusion": (
        "The CRAFT prompt produced more consistent structured output "
        "and passed Pydantic validation. The basic prompt identified "
        "similar reasons but failed the predefined schema."
    )
}

with open("prompt_comparison_report.json", "w") as f:
    json.dump(comparison, f, indent=4)

print("Prompt comparison report saved successfully!")

Prompt comparison report saved successfully!


In [22]:
import os
import json
import zipfile

from pathlib import Path

# Create submission folder
output_dir = Path("CareLoop_Part4_Submission")
output_dir.mkdir(exist_ok=True)

# Save the raw basic prompt response, even though validation failed
with open(output_dir / "basic_prompt_response.json", "w") as f:
    f.write(basic_response.text)

# Save the comparison report
with open(output_dir / "prompt_comparison_report.json", "w") as f:
    json.dump(comparison, f, indent=4)

# Create README
readme = """
# CareLoop Clinics — Part 4: Generative AI Analysis

## Overview
Gemini Flash was used to analyse the first 10 patient follow-up notes.

## Task 4.1 — Gemini API and CRAFT Prompt
- Integrated Gemini Flash using the google-genai SDK.
- Prepared a CRAFT prompt (Context, Role, Action, Format, Tone).
- Analysed 10 follow-up notes.
- Saved the raw Gemini response.

## Task 4.2 — Pydantic Validation
- Created Pydantic models for structured responses.
- Validated all 10 CRAFT results.
- Generated notes_analysed.csv.
- Handled the empty N004 note as Unknown.

## Task 4.3 — Validation and Prompt Comparison
- Tested two deliberately invalid responses.
- Both were rejected by Pydantic.
- Compared CRAFT and basic prompt outputs.
- CRAFT output passed schema validation.
- Basic output used inconsistent reason labels and failed
  strict schema validation.
- Both raw outputs and the comparison report are included.

## Task 4.4 — Clinic Analysis
- Merged notes with appointments and clinics.
- All 10 appointment and clinic records matched.
- Generated reason-by-clinic and reason-by-city tables.
- Generated clinic-specific recommendations.

## Reason Summary
- No reminder: 2
- Transport or distance: 2
- Work or scheduling: 2
- Financial: 1
- Illness: 1
- Contact issue: 1
- Unknown: 1

These are findings from the first 10 notes only.
They are not population-wide clinic rates.

## Files
- gemini_raw_response.json
- basic_prompt_response.json
- notes_analysed.csv
- merged_notes_appointments_clinics.csv
- reason_by_clinic.csv
- reason_by_city.csv
- clinic_specific_recommendations.csv
- prompt_comparison_report.json

## Security
Never publish API keys or credentials.
Use Colab Secrets for API credentials.
"""

(output_dir / "README.md").write_text(readme)

# Copy generated output files into submission folder
files_to_copy = [
    "gemini_raw_response.json",
    "notes_analysed.csv",
    "merged_notes_appointments_clinics.csv",
    "reason_by_clinic.csv",
    "reason_by_city.csv",
    "clinic_specific_recommendations.csv",
    "prompt_comparison_report.json"
]

for filename in files_to_copy:
    if os.path.exists(filename):
        with open(filename, "rb") as src:
            (output_dir / filename).write_bytes(src.read())
    else:
        print("WARNING — file not found:", filename)

# Create ZIP
zip_name = "CareLoop_Part4_Submission.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for file in output_dir.iterdir():
        if file.is_file():
            z.write(file, arcname=f"CareLoop_Part4_Submission/{file.name}")

print("\nPART 4 PACKAGE CREATED")
print("Files included:")
for file in sorted(output_dir.iterdir()):
    print("-", file.name)

print("\nZIP:", zip_name)
print("Size:", round(os.path.getsize(zip_name) / 1024, 2), "KB")


PART 4 PACKAGE CREATED
Files included:
- README.md
- basic_prompt_response.json
- clinic_specific_recommendations.csv
- gemini_raw_response.json
- merged_notes_appointments_clinics.csv
- notes_analysed.csv
- prompt_comparison_report.json
- reason_by_city.csv
- reason_by_clinic.csv

ZIP: CareLoop_Part4_Submission.zip
Size: 8.48 KB
